# 02 · Training

Allena un modello su un dataset con una augmentation, per uno o più seed.
L'epoca migliore si sceglie solo con l'EER sul dev; i test sono nel notebook 03.

**Attenzione:** il training gira dentro il notebook, quindi VS Code deve restare aperto (e il PC non deve andare in sospensione).
Se si interrompe, basta rilanciare la cella: riparte dall'ultima epoca finita. I seed già finiti vengono saltati.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))   # per importare la cartella tesi/ del progetto

import json
import pandas as pd
import matplotlib.pyplot as plt
import torch

from tesi import training, risultati
from tesi.modelli import IPERPARAMETRI

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nessuna, si usa la CPU")

## 0. Esperimenti già fatti con il codice vecchio (una volta sola)

Porta i risultati dei primi esperimenti su ASVspoof 5 (in `artifacts/runs/`) nella cartella nuova `artifacts/esperimenti/`:
pesi del modello migliore, punteggi su dev ed eval e log. Così si possono testare anche sugli altri dataset senza rifare il training.
Se una cartella vecchia ha un nome diverso, cambialo qui.

In [ ]:
risultati.importa_run_vecchia("mambo_asvspoof5", "mambo", "nessuna")
risultati.importa_run_vecchia("mambo_rawboost_asvspoof5", "mambo", "rawboost")
risultati.importa_run_vecchia("nes2net_x_rawboost_asvspoof5", "nes2net", "rawboost")

## 1. Scelta dell'esperimento

- `MODELLO`: `"mambo"` (XLSR-MamBo), `"nes2net"` (Nes2Net-X), `"mhfa"` (WavLM Base + MHFA), `"tfparn"` (TFPARN)
- `DATASET`: `"asvspoof5"` (sottoinsieme da 10.000 clip) oppure `"asvspoof2019"` (2019 LA intero)
- `AUGMENTATION`: `"nessuna"`, `"rawboost"`, `"rumore"`, `"codec"`; per TFPARN anche `"specaugment"`

Ordine proposto:
1. MHFA e TFPARN con RawBoost su ASVspoof 5 (MamBo e Nes2Net-X ci sono già);
2. i quattro modelli con RawBoost su 2019 LA;
3. confronto delle augmentation (nessuna, rumore, codec) su MHFA e TFPARN, su entrambi i dataset.

Tempi indicativi sulla 3060 per un seed: MamBo ~2 h, Nes2Net-X ~3 h, MHFA ~1 h, TFPARN meno di 1 h su ASVspoof 5;
su 2019 LA circa 2,5 volte tanto.

In [ ]:
MODELLO = "mhfa"
DATASET = "asvspoof5"
AUGMENTATION = "rawboost"
SEEDS = [1, 2, 3]

IPERPARAMETRI[MODELLO]

In [ ]:
for seed in SEEDS:
    training.addestra(MODELLO, DATASET, AUGMENTATION, seed)

## 2. Curve di training

Loss di train ed EER sul dev per ogni epoca. Si può lanciare anche mentre il training è in corso (da un altro notebook).

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
for seed in SEEDS:
    file_log = training.cartella_esperimento(MODELLO, DATASET, AUGMENTATION, seed) / "log.csv"
    if file_log.exists():
        log = pd.read_csv(file_log)
        ax1.plot(log["epoca"], log["loss_train"], marker="o", label=f"seed {seed}")
        ax2.plot(log["epoca"], log["eer_dev"], marker="o", label=f"seed {seed}")
ax1.set(title="Loss di train", xlabel="epoca")
ax2.set(title="EER sul dev (%)", xlabel="epoca")
ax1.legend()
fig.suptitle(f"{MODELLO} · {DATASET} · {AUGMENTATION}")
fig.tight_layout()